In [4]:
import pandas as pd
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [5]:
# Load datasets
sym_df = pd.read_csv("DiseaseAndSymptoms.csv")
pre_df = pd.read_csv("Disease precaution.csv")

In [6]:
print(f"Symptoms dataset: {sym_df.shape}")
print(f"Precaution dataset: {pre_df.shape}")

Symptoms dataset: (4920, 18)
Precaution dataset: (41, 5)


In [7]:
# Clean disease names (strip whitespace)
sym_df['Disease'] = sym_df['Disease'].str.strip()
pre_df['Disease'] = pre_df['Disease'].str.strip()

In [9]:
# Combine all symptom columns into one list per row
symptom_cols = [c for c in sym_df.columns if c.startswith('Symptom')]

def get_symptoms(row):
    symptoms = []
    for col in symptom_cols:
        val = row[col]
        if pd.notna(val):
            symptoms.append(str(val).strip().replace(' ', '_').lower())
    return symptoms

sym_df['symptom_list'] = sym_df.apply(get_symptoms, axis=1)

In [10]:
# Get all unique symptoms for the vocabulary
all_symptoms = set()
for s_list in sym_df['symptom_list']:
    all_symptoms.update(s_list)
all_symptoms = sorted(list(all_symptoms))
print(f"Total unique symptoms: {len(all_symptoms)}")

Total unique symptoms: 131


In [11]:
# MultiLabelBinarizer — converts symptom list to binary vector
mlb = MultiLabelBinarizer(classes=all_symptoms)
X = mlb.fit_transform(sym_df['symptom_list'])
print(f"Feature matrix shape: {X.shape}")

Feature matrix shape: (4920, 131)


In [12]:
# Encode disease labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(sym_df['Disease'])
print(f"Classes: {label_encoder.classes_}")

Classes: ['(vertigo) Paroymsal  Positional Vertigo' 'AIDS' 'Acne'
 'Alcoholic hepatitis' 'Allergy' 'Arthritis' 'Bronchial Asthma'
 'Cervical spondylosis' 'Chicken pox' 'Chronic cholestasis' 'Common Cold'
 'Dengue' 'Diabetes' 'Dimorphic hemmorhoids(piles)' 'Drug Reaction'
 'Fungal infection' 'GERD' 'Gastroenteritis' 'Heart attack' 'Hepatitis B'
 'Hepatitis C' 'Hepatitis D' 'Hepatitis E' 'Hypertension'
 'Hyperthyroidism' 'Hypoglycemia' 'Hypothyroidism' 'Impetigo' 'Jaundice'
 'Malaria' 'Migraine' 'Osteoarthristis' 'Paralysis (brain hemorrhage)'
 'Peptic ulcer diseae' 'Pneumonia' 'Psoriasis' 'Tuberculosis' 'Typhoid'
 'Urinary tract infection' 'Varicose veins' 'hepatitis A']


In [14]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Training: {X_train.shape[0]}, Testing: {X_test.shape[0]}")

Training: 3936, Testing: 984


In [15]:
# Train Random Forest
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
print("Model trained!")

Model trained!


In [16]:
# Evaluate
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"\n✅ Accuracy: {accuracy * 100:.2f}%")
print(classification_report(y_test, y_pred,
      target_names=label_encoder.classes_, zero_division=0))


✅ Accuracy: 100.00%
                                         precision    recall  f1-score   support

(vertigo) Paroymsal  Positional Vertigo       1.00      1.00      1.00        18
                                   AIDS       1.00      1.00      1.00        30
                                   Acne       1.00      1.00      1.00        24
                    Alcoholic hepatitis       1.00      1.00      1.00        25
                                Allergy       1.00      1.00      1.00        24
                              Arthritis       1.00      1.00      1.00        23
                       Bronchial Asthma       1.00      1.00      1.00        33
                   Cervical spondylosis       1.00      1.00      1.00        23
                            Chicken pox       1.00      1.00      1.00        21
                    Chronic cholestasis       1.00      1.00      1.00        15
                            Common Cold       1.00      1.00      1.00        23
      

In [17]:
# Save everything
pickle.dump(model, open("disease_model.pkl", "wb"))
pickle.dump(mlb, open("mlb.pkl", "wb"))
pickle.dump(label_encoder, open("label_encoder.pkl", "wb"))
print("\n✅ All models saved!")


✅ All models saved!
